In [1]:
import amulet
import os
import json
import numpy as np
from amulet import load_level
from amulet.api.selection import SelectionBox
import itertools

INFO - PyMCTranslate Version 378


In [2]:
data_path = '../data/processed_builds/'
samples = os.listdir(data_path)
samples = [i for i in samples if i.endswith('.schematic') or i.endswith('.schem')]

# filename = "build_batch_533_13840_4.schem"
# filename = "build_batch_257_6676_19.schem"
# filename = "build_batch_220_5711_1.schem"
# filename = "build_batch_181_4702_9.schem"
# filename = "build_batch_181_4702_1.schem"
# filename = "build_batch_162_4207_2.schem"
# filename = "build_batch_332_8611_1.schem"
filename = "build_batch_104_2682_1.schem"
# filename = "build_batch_257_6676_19.schem"
# filename = "build_batch_257_6676_19.schem"
# filename = "build_batch_257_6676_19.schem"


example = os.path.join(data_path, filename)

# took 71 minutes for 2100 I think

In [3]:
build = load_level(example)

# Assert that theres only one dimension
dim = build.dimensions[0]

# Assert no negative values
bounds = build.bounds(dim).bounds_array
# print(bounds)

dims = [ bounds[1,i] - bounds[0,i] for i in range(3)]
# print(dims)

build_array = np.zeros(dims, dtype=np.int16)

universal_block_palette = {}
universal_block_objects = []
java_block_palette = {}
java_block_objects = []

universal_token_counter = 0
java_token_counter = 0

for x in range(dims[0]):
    for y in range(dims[1]):
        for z in range(dims[2]):
            # Get the block string
            universal_block = build.get_block( x, y, z, dim)
            java_block = build.get_version_block( x, y, z, dim, ('java', (1, 21, 5)))[0]
            
            universal_block_str = str(universal_block)
            java_block_str = str(java_block.blockstate)
            
            # Add to palette if it isn't already
            if universal_block_str not in universal_block_palette.keys():
                universal_block_palette[universal_block_str] = universal_token_counter
                universal_block_objects.append(universal_block)
                universal_token_counter += 1
            
            if java_block_str not in java_block_palette.keys():
                java_block_palette[java_block_str] = java_token_counter
                java_block_objects.append(java_block)
                java_token_counter += 1
            
            # Set this voxel in the build array
            build_array[x,y,z] = java_block_palette[java_block_str]

java_tok2block = {v:k for k,v in java_block_palette.items()}


for key in java_block_palette.keys():
    print(f'{key}: {java_block_palette[key]}')

# print()

# for block in java_block_objects:
#     print(block)

# print()
# for key in universal_block_palette.keys():
#     print(f'{key}: {universal_block_palette[key]}')

# print()

# for block in universal_block_objects:
#     print(block)

INFO - Loading level ../data/processed_builds/build_batch_104_2682_1.schem


minecraft:dirt: 0
minecraft:grass_block[snowy=false]: 1
minecraft:air: 2
minecraft:smooth_stone: 3
minecraft:cyan_terracotta: 4
minecraft:light_gray_stained_glass_pane[east=true,north=false,south=true,west=false]: 5
minecraft:light_gray_stained_glass_pane[east=false,north=true,south=true,west=false]: 6
minecraft:light_gray_stained_glass_pane[east=true,north=true,south=false,west=false]: 7
minecraft:purpur_block: 8
minecraft:blue_terracotta: 9
minecraft:light_gray_stained_glass: 10
minecraft:quartz_block: 11
minecraft:purpur_stairs[facing=west,half=bottom,shape=straight]: 12
minecraft:purpur_stairs[facing=south,half=bottom,shape=inner_right]: 13
minecraft:light_gray_stained_glass_pane[east=true,north=false,south=false,west=true]: 14
minecraft:purpur_stairs[facing=north,half=bottom,shape=inner_left]: 15
minecraft:light_gray_stained_glass_pane[east=false,north=false,south=true,west=true]: 16
minecraft:sea_lantern: 17
minecraft:purpur_stairs[facing=south,half=bottom,shape=straight]: 18
min

In [4]:
import mcschematic
import numpy as np

def array_to_schematic(token_array, tok2block, save_path, filename="my_schematic"):
    """
    Converts a 3D numpy array of integer tokens into a Minecraft schematic file.

    Args:
        token_array (np.ndarray): A 3D array where values correspond to block types.
        filename (str): The name of the output schematic file (without extension).
    """
    schem = mcschematic.MCSchematic()

    d, h, w = token_array.shape

    # Iterate through the 3D array and place blocks
    for x in range(d):
        for y in range(h):
            for z in range(w):
                token = token_array[x, y, z]
                # block_name = tok2block.get(f'{token}', "minecraft:air") # Default to air if token not found
                block_name = tok2block[token]
                

                # Place the block in the schematic at the specified coordinates (x, y, z)
                schem.setBlock((x, y, z), block_name)

    # Save the schematic file
    schem.save(save_path, filename, mcschematic.Version.JE_1_21_5, True)
    
    print(f"Successfully saved schematic to {filename}.schem")


array_to_schematic(build_array, java_tok2block, 'saved_schems/', 'test_schem')


Successfully saved schematic to test_schem.schem


In [5]:
java_tok2block

{0: 'minecraft:stone',
 1: 'minecraft:dirt',
 2: 'minecraft:air',
 3: 'minecraft:sandstone',
 4: 'minecraft:granite',
 5: 'minecraft:sand',
 6: 'minecraft:cactus[age=0]',
 7: 'minecraft:cactus[age=1]',
 8: 'minecraft:iron_ore',
 9: 'minecraft:coal_ore',
 10: 'minecraft:andesite',
 11: 'minecraft:dead_bush',
 12: 'minecraft:diorite',
 13: 'minecraft:lava[level=0]',
 14: 'minecraft:grass_block[snowy=false]',
 15: 'minecraft:short_grass',
 16: 'minecraft:acacia_leaves[distance=7,persistent=false]',
 17: 'minecraft:acacia_log[axis=y]',
 18: 'minecraft:water[level=0]',
 19: 'minecraft:sugar_cane[age=0]',
 20: 'minecraft:gravel',
 21: 'minecraft:clay'}

In [6]:
np.unique(build_array, return_counts=True)

(array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
        17, 18, 19, 20, 21], dtype=int16),
 array([ 6780,  1086, 53854, 14178,   368, 12769,     9,     1,    49,
           70,   221,     6,   303,    19,   355,   158,    80,    12,
         1807,     5,    26,     4], dtype=int64))